# DOST RoBERTa for Tagalog NSFW word detection

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, average_precision_score, confusion_matrix
)
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import sys
import logging

# NOTE: transformers / datasets are imported in the install cell below, AFTER
# install_requirements() runs, because they are not preinstalled on a fresh Colab.


class Logger:
    def __init__(self):
        self.logger = logging.getLogger("NSFW_Model")
        self.logger.setLevel(logging.INFO)
        # Don't propagate to the root logger (Colab attaches its own handler,
        # which otherwise prints every line a second time as "INFO:NSFW_Model:...").
        self.logger.propagate = False
        # Avoid duplicate handlers in Colab
        if not self.logger.handlers:
            handler = logging.StreamHandler(sys.stdout)
            handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s', datefmt='%H:%M:%S'))
            self.logger.addHandler(handler)

    def set_log_file(self, path):
        handler = logging.FileHandler(path)
        handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
        self.logger.addHandler(handler)
        self.info(f"Logging to file: {path}")

    def info(self, message):
        self.logger.info(message)

    def warning(self, message):
        self.logger.warning(message)

    def error(self, message):
        self.logger.error(message)

    def exception(self, message):
        self.logger.exception(message)

    def section(self, title):
        self.info("\n" + "="*70)
        self.info(title.upper())
        self.info("="*70)


logger = Logger()

Check Hardware

In [2]:
class Config:
    def __init__(self):
        # Metadata
        self.model_version = "1.0.0"

        # Base model (DOST-ASTI Tagalog RoBERTa)
        self.model_name = "dost-asti/RoBERTa-tl-sentiment-analysis"

        # Model Hyperparameters
        self.max_length = 128

        # Training Hyperparameters
        self.learning_rate = 2e-5
        self.weight_decay = 0.01
        self.batch_size = 16   # auto-bumped to 32 when a GPU is detected
        self.epochs = 5

        # Dataset Split (70/15/15)
        self.validation_split = 0.15
        self.test_split = 0.15

        self.log_config()

    def log_config(self):
        """Print current configuration"""
        logger.section("MODEL CONFIGURATION")
        logger.info(f"Model Version: {self.model_version}")
        logger.info(f"Base Model: {self.model_name}")
        logger.info(f"Max Sequence Length: {self.max_length}")
        logger.info(f"Batch Size: {self.batch_size}")
        logger.info(f"Epochs: {self.epochs}")
        logger.info(f"Learning Rate: {self.learning_rate}")
        logger.info(f"Weight Decay: {self.weight_decay}")
        logger.info(f"Validation Split: {self.validation_split}")
        logger.info(f"Test Split: {self.test_split}")

In [3]:
def install_requirements():
    required_packages = [
        'transformers[torch]',
        'datasets',
        'torch',
        'pandas',
        'scikit-learn',
        'matplotlib',
        'seaborn',
        'onnx',
        'onnxruntime',
        'onnxscript',    # required for newer torch.onnx export
        'litert-torch',  # PyTorch -> LiteRT/TFLite (formerly 'ai-edge-torch', now renamed)
        'torchao',       # PT2E quantization flow (moved out of torch.ao in newer torch)
    ]

    # Map pip package names to their import names where they differ
    import_name_map = {
        'scikit-learn': 'sklearn',
        'transformers[torch]': 'transformers',
        'litert-torch': 'litert_torch',
    }

    for package in required_packages:
        import_name = import_name_map.get(package, package.split('[')[0])
        try:
            __import__(import_name)
        except ImportError:
            print(f"Installing {package}...")
            # Non-fatal: a single bad/renamed package must not abort the whole setup.
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            except subprocess.CalledProcessError as e:
                print(f"WARNING: failed to install {package}: {e}")

In [4]:
print("Checking and installing dependencies...")
install_requirements()

# Import heavy deps AFTER installation (not preinstalled on a fresh Colab)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset

print("All dependencies loaded successfully!")

Checking and installing dependencies...
Installing onnx...
Installing onnxruntime...
Installing onnxscript...
Installing litert-torch...


All dependencies loaded successfully!


# LOAD AND PREPARE DATASET

In [5]:
def load_dataset(csv_path='cleaned_data.csv'):
    logger.info(f"Loading dataset from {csv_path}...")

    # Attempt to read CSV directly. Will raise FileNotFoundError if missing.
    df = pd.read_csv(csv_path)
    logger.info(f"Dataset loaded successfully. Shape: {df.shape}")

    # Validate required columns
    required_cols = ['text', 'label']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Clean and validate data
    df = df.dropna(subset=['text', 'label'])
    df['text'] = df['text'].astype(str).str.strip()

    # Map text labels to integers if needed (safe->0, nsfw->1)
    if df['label'].dtype == 'object':
        logger.info("Mapping text labels (safe/nsfw) to binary integers...")
        label_map = {'safe': 0, 'nsfw': 1}
        df['label'] = df['label'].astype(str).str.lower().str.strip().map(label_map)

        if df['label'].isnull().any():
            logger.warning(f"Dropping {df['label'].isnull().sum()} rows with unknown labels")
            df = df.dropna(subset=['label'])

    df['label'] = df['label'].astype(int)

    # Remove empty texts
    df = df[df['text'].str.len() > 0]

    # Validate labels are binary
    unique_labels = df['label'].unique()
    if not all(label in [0, 1] for label in unique_labels):
        raise ValueError("Labels must be 0 (safe) or 1 (nsfw)")

    logger.info(f"Dataset validation complete. Clean shape: {df.shape}")
    logger.info(f"Label distribution:\n{df['label'].value_counts()}")

    return df

# TOKENIZATION AND PREPROCESSING

In [6]:
def preprocess_data(df, config):
    """Split 70/15/15 (stratified) and tokenize with the RoBERTa tokenizer."""
    logger.info("Preprocessing data...")

    X = df['text'].values
    y = df['label'].values

    # 1st Split: Train vs (Val + Test)
    rest_size = config.validation_split + config.test_split
    X_train_raw, X_rest_raw, y_train, y_rest = train_test_split(
        X, y, test_size=rest_size, random_state=42, stratify=y
    )

    # 2nd Split: Val vs Test (relative size out of the remaining portion)
    test_size_relative = config.test_split / rest_size if rest_size > 0 else 0
    X_val_raw, X_test_raw, y_val, y_test = train_test_split(
        X_rest_raw, y_rest, test_size=test_size_relative,
        random_state=42, stratify=y_rest
    )

    logger.info(f"Training samples: {len(X_train_raw)}")
    logger.info(f"Validation samples: {len(X_val_raw)}")
    logger.info(f"Test samples: {len(X_test_raw)}")

    # Load tokenizer
    logger.info(f"Loading tokenizer: {config.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    def make_dataset(texts, labels):
        return Dataset.from_dict({'text': list(texts), 'labels': list(labels)})

    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            padding=False,          # handled dynamically by the data collator
            max_length=config.max_length
        )

    logger.info("Tokenizing datasets...")
    train_dataset = make_dataset(X_train_raw, y_train).map(tokenize_function, batched=True)
    val_dataset = make_dataset(X_val_raw, y_val).map(tokenize_function, batched=True)
    test_dataset = make_dataset(X_test_raw, y_test).map(tokenize_function, batched=True)

    logger.info("Tokenization complete.")

    return (train_dataset, val_dataset, test_dataset,
            np.array(y_train), np.array(y_val), np.array(y_test),
            tokenizer, X_test_raw)

# ROBERTA MODEL ARCHITECTURE

In [7]:
def build_roberta_model(config):
    """Load the DOST RoBERTa-tl model with a fresh 2-class classification head."""
    logger.info("Building RoBERTa model (transfer learning)...")

    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=2,
        id2label={0: "Safe", 1: "NSFW"},
        label2id={"Safe": 0, "NSFW": 1},
        ignore_mismatched_sizes=True   # base model ships a 3-class head; reinit for binary
    )

    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"Model loaded: {config.model_name}")
    logger.info(f"Total parameters: {n_params:,}")

    return model

# MODEL TRAINING

In [8]:
def _history_from_logs(log_history):
    """Convert HF Trainer log_history into {accuracy, val_accuracy, loss, val_loss} per epoch."""
    train_loss_by_epoch = {}
    val_loss_by_epoch = {}
    val_acc_by_epoch = {}

    for entry in log_history:
        epoch = entry.get('epoch')
        if epoch is None:
            continue
        if 'loss' in entry:          # training-step log (logged mid-epoch)
            # A step at fractional epoch e belongs to training epoch ceil(e),
            # so it lines up with the per-epoch eval logged at the integer boundary.
            ep = max(1, int(np.ceil(epoch)))
            train_loss_by_epoch.setdefault(ep, []).append(entry['loss'])
        if 'eval_loss' in entry:     # evaluation log (once per epoch, integer epoch)
            val_loss_by_epoch[int(round(epoch))] = entry['eval_loss']
        if 'eval_accuracy' in entry:
            val_acc_by_epoch[int(round(epoch))] = entry['eval_accuracy']

    epochs = sorted(val_loss_by_epoch.keys())
    history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}
    for ep in epochs:
        losses = train_loss_by_epoch.get(ep, [])
        history['loss'].append(float(np.mean(losses)) if losses else float('nan'))
        history['val_loss'].append(val_loss_by_epoch.get(ep, float('nan')))
        history['val_accuracy'].append(val_acc_by_epoch.get(ep, float('nan')))
        # HF does not log train accuracy by default; placeholder keeps columns aligned
        history['accuracy'].append(float('nan'))
    return history


def train_model(model, train_dataset, val_dataset, tokenizer, config, output_dir):
    logger.info("\nStarting model training...")
    os.makedirs(output_dir, exist_ok=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Hardware-adaptive settings (target environment: Colab A100 GPU)
    if torch.cuda.is_available():
        logger.info(f"GPU detected: {torch.cuda.get_device_name(0)}")
        use_fp16 = True
        batch_size = 32
        num_workers = 2
    else:
        logger.info("No GPU detected. Using CPU (slower).")
        use_fp16 = False
        batch_size = config.batch_size
        num_workers = 0

    # Log the EFFECTIVE settings (Config.log_config prints the pre-override defaults).
    logger.info(f"Effective batch size: {batch_size} | fp16: {use_fp16} | dataloader workers: {num_workers}")

    training_args_dict = {
        'output_dir': output_dir,
        'num_train_epochs': config.epochs,
        'per_device_train_batch_size': batch_size,
        'per_device_eval_batch_size': batch_size,
        'learning_rate': config.learning_rate,
        'weight_decay': config.weight_decay,
        'logging_steps': 10,
        'save_total_limit': 2,
        'load_best_model_at_end': True,
        'metric_for_best_model': "eval_loss",
        'greater_is_better': False,
        'report_to': [],
        'seed': 42,
        'dataloader_num_workers': num_workers,
        'remove_unused_columns': True,
        'fp16': use_fp16,  # Mixed precision on GPU
    }

    # Handle the transformers eval_strategy / evaluation_strategy rename
    if hasattr(TrainingArguments, 'eval_strategy'):
        training_args_dict['eval_strategy'] = "epoch"
        training_args_dict['save_strategy'] = "epoch"
    else:
        training_args_dict['evaluation_strategy'] = "epoch"
        training_args_dict['save_strategy'] = "epoch"

    training_args = TrainingArguments(**training_args_dict)

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {'accuracy': accuracy_score(labels, predictions)}

    # Trainer renamed `tokenizer` -> `processing_class` in transformers >= 4.46
    import inspect
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    if 'processing_class' in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    logger.info("Training in progress...")
    trainer.train()

    # Save the best model + tokenizer (single source of truth on disk)
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    logger.info(f"Model saved to {output_dir}")

    # --- Save Training History to CSV (Keras-style dict from HF logs) ---
    history = _history_from_logs(trainer.state.log_history)
    history_df = pd.DataFrame(history)
    history_csv_path = os.path.join(output_dir, 'training_history.csv')
    history_df.to_csv(history_csv_path, index_label='epoch')
    logger.info(f"Training history saved to {history_csv_path}")
    # --------------------------------------------------------------------

    return trainer, history

# MODEL EVALUATION AND VISUALIZATION

In [9]:
def _softmax(logits):
    """Numerically stable row-wise softmax for a (n, num_labels) logit array."""
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=1, keepdims=True)


def evaluate_model(trainer, val_dataset, y_val, output_dir,
                   test_dataset=None, y_test=None, threshold=None):
    """Evaluate model and generate metrics (mirrors the LSTM/BiLSTM pipeline).

    If `threshold` is provided, it is reused directly (no re-tuning / extra
    validation prediction); otherwise the optimal threshold is tuned on the
    validation set.
    """
    logger.info("\nEvaluating model...")

    # --- Threshold Tuning (Using Validation Set) ---
    # The optimal threshold is always calculated on the VALIDATION set to avoid leakage.
    val_prob = None  # cached P(NSFW) on the validation set; reused for the val report below
    if threshold is not None:
        best_threshold = threshold
        logger.info(f"Reusing calibrated threshold: {best_threshold:.4f}")
    else:
        logger.info("Tuning decision threshold on Validation Set...")
        val_logits = trainer.predict(val_dataset).predictions
        val_prob = _softmax(val_logits)[:, 1]   # P(NSFW)
        y_val_prob = val_prob

        # Search a sane band, only deviating from the neutral 0.5 if the validation
        # gain is large enough to be real rather than noise.
        thresholds = np.arange(0.30, 0.71, 0.01)
        f1_scores = [f1_score(y_val, (y_val_prob > t).astype(int)) for t in thresholds]
        best_idx = int(np.argmax(f1_scores))
        tuned_threshold = float(thresholds[best_idx])
        tuned_f1 = f1_scores[best_idx]

        f1_at_half = f1_score(y_val, (y_val_prob > 0.5).astype(int))
        if tuned_f1 - f1_at_half > 0.01:
            best_threshold, best_nsfw_f1 = tuned_threshold, tuned_f1
        else:
            best_threshold, best_nsfw_f1 = 0.5, f1_at_half

        logger.info(f"Optimal Threshold: {best_threshold:.4f} (tuned {tuned_threshold:.2f}->F1 {tuned_f1:.4f} vs 0.50->F1 {f1_at_half:.4f})")
        logger.info(f"Validation NSFW F1: {best_nsfw_f1:.4f}")

    # --- Final Evaluation (Test set if provided, else Validation) ---
    if test_dataset is not None and y_test is not None:
        logger.info("Calculating final metrics on Test Set using optimal threshold...")
        eval_dataset = test_dataset
        y_eval = y_test
        eval_name = "Test"
        y_pred_prob_flat = _softmax(trainer.predict(eval_dataset).predictions)[:, 1]
    else:
        logger.info("Calculating final metrics on Validation Set...")
        eval_dataset = val_dataset
        y_eval = y_val
        eval_name = "Validation"
        # Reuse the probabilities already computed during threshold tuning (avoids a
        # second full forward pass over the same validation set).
        if val_prob is not None:
            y_pred_prob_flat = val_prob
        else:
            y_pred_prob_flat = _softmax(trainer.predict(eval_dataset).predictions)[:, 1]

    y_pred = (y_pred_prob_flat > best_threshold).astype(int)

    # When the threshold was tuned AND scored on the same (validation) set, the
    # resulting metrics are optimistically biased. The Test set is the unbiased number.
    tuned_on_this_set = (eval_name == "Validation" and threshold is None)

    # Calculate metrics
    accuracy = accuracy_score(y_eval, y_pred)
    # ROC-AUC / AUPRC are undefined with a single class present; guard against it.
    if len(np.unique(y_eval)) < 2:
        logger.warning(f"{eval_name} set has only one class present; ROC-AUC/AUPRC set to NaN.")
        roc_auc = float('nan')
        auprc = float('nan')
    else:
        roc_auc = roc_auc_score(y_eval, y_pred_prob_flat)
        auprc = average_precision_score(y_eval, y_pred_prob_flat)
    macro_f1 = f1_score(y_eval, y_pred, average='macro')
    macro_precision = precision_score(y_eval, y_pred, average='macro')
    macro_recall = recall_score(y_eval, y_pred, average='macro')
    nsfw_f1 = f1_score(y_eval, y_pred)   # Binary default is pos_label=1

    report = classification_report(y_eval, y_pred, target_names=['Safe', 'NSFW'], digits=4)
    cm = confusion_matrix(y_eval, y_pred)

    bias_note = ("\nNOTE: threshold was tuned AND scored on this same validation set,\n"
                 "so these numbers are optimistically biased. Use the TEST set as the\n"
                 "unbiased estimate of generalization performance.\n") if tuned_on_this_set else ""

    # Save metrics text
    metrics_path = os.path.join(output_dir, f'{eval_name.lower()}_evaluation_metrics.txt')
    metrics_text = f"""DOST RoBERTa Tagalog NSFW Detection Model Evaluation
{'='*70}

Model Architecture: RoBERTa-tl (DOST-ASTI, fine-tuned)
Task: Binary Classification (Safe vs NSFW)
Evaluation Set: {eval_name}
{bias_note}
Calibration:
----------------------------------------------------------------------
Optimized Threshold: {best_threshold:.4f} (Tuned on Validation)
Target Metric:      Max F1 (NSFW)
----------------------------------------------------------------------

Key Metrics ({eval_name}):
----------------------------------------------------------------------
Overall Accuracy:   {accuracy:.4f}
Macro Precision:    {macro_precision:.4f}
Macro Recall:       {macro_recall:.4f}
Macro F1-Score:     {macro_f1:.4f}
NSFW F1-Score:      {nsfw_f1:.4f}
AUROC:              {roc_auc:.4f}
PR-AUC (AUPRC):     {auprc:.4f}
----------------------------------------------------------------------

Classification Report:
{report}

Confusion Matrix:
{cm}
"""
    with open(metrics_path, 'w') as f:
        f.write(metrics_text)

    # --- Save Metrics to CSV ---
    metrics_data = {
        'set': eval_name,
        'accuracy': accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'nsfw_f1': nsfw_f1,
        'roc_auc': roc_auc,
        'pr_auc': auprc,
        'optimal_threshold': best_threshold,
        'threshold_tuned_on_this_set': tuned_on_this_set
    }
    metrics_csv_path = os.path.join(output_dir, f'{eval_name.lower()}_metrics.csv')
    pd.DataFrame([metrics_data]).to_csv(metrics_csv_path, index=False)
    logger.info(f"{eval_name} metrics saved to {metrics_csv_path}")

    # Save detailed classification report to CSV
    report_dict = classification_report(y_eval, y_pred, target_names=['Safe', 'NSFW'], output_dict=True)
    report_csv_path = os.path.join(output_dir, f'{eval_name.lower()}_classification_report.csv')
    pd.DataFrame(report_dict).transpose().to_csv(report_csv_path)
    logger.info(f"{eval_name} classification report saved to {report_csv_path}")

    # --- Save Detailed Predictions to CSV ---
    predictions_df = pd.DataFrame({
        'y_true': y_eval,
        'y_pred': y_pred,
        'y_prob': y_pred_prob_flat
    })
    predictions_csv_path = os.path.join(output_dir, f'{eval_name.lower()}_predictions.csv')
    predictions_df.to_csv(predictions_csv_path, index=False)
    logger.info(f"{eval_name} predictions saved to {predictions_csv_path}")
    # ---------------------------

    if tuned_on_this_set:
        logger.warning("Validation metrics above are threshold-tuned on this same set (optimistic). "
                       "Refer to the Test set for the unbiased result.")
    logger.info(f"{eval_name} Accuracy: {accuracy:.4f}")
    logger.info(f"{eval_name} Macro F1-Score: {macro_f1:.4f}")
    logger.info(f"Metrics saved to {metrics_path}")

    # Plot confusion matrix
    plot_confusion_matrix(cm, output_dir, filename=f'{eval_name.lower()}_confusion_matrix.png')

    return accuracy, report, best_threshold

In [10]:
def plot_confusion_matrix(cm, output_dir, filename='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Safe', 'NSFW'],
        yticklabels=['Safe', 'NSFW']
    )
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path)
    logger.info(f"Confusion matrix saved to {save_path}")
    plt.close()

In [11]:
def plot_training_history(history, output_dir):
    """Plot training history (accepts a dict of per-epoch metric lists)."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss plot
    if history.get('loss'):
        axes[0].plot(history['loss'], label='Train Loss', marker='o')
    if history.get('val_loss'):
        axes[0].plot(history['val_loss'], label='Val Loss', marker='o')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    # Accuracy plot (HF logs validation accuracy per epoch)
    train_acc = history.get('accuracy', [])
    if train_acc and any(not np.isnan(a) for a in train_acc):
        axes[1].plot(train_acc, label='Train Accuracy', marker='o')
    if history.get('val_accuracy'):
        axes[1].plot(history['val_accuracy'], label='Val Accuracy', marker='o')
    axes[1].set_title('Model Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'))
    logger.info(f"Training history saved to {output_dir}/training_history.png")
    plt.close()

# MODEL EXPORT

In [12]:
def save_artifacts(trainer, tokenizer, config, output_dir, threshold=0.5):
    """Save deployment artifacts (config + ONNX/TFLite).

    The model and tokenizer are already written to `output_dir` by train_model(),
    so they are not re-saved here.
    """
    logger.info("\nSaving model artifacts...")

    # Save deployment config (incl. calibrated decision threshold).
    # NOTE: written as 'deployment_config.json' to avoid clobbering the model's own config.json.
    config_dict = {
        'model_version': config.model_version,
        'base_model': config.model_name,
        'decision_threshold': float(threshold),
        'max_length': config.max_length,
        'num_labels': 2,
        'id2label': {0: "Safe", 1: "NSFW"}
    }
    config_path = os.path.join(output_dir, 'deployment_config.json')
    with open(config_path, 'w') as f:
        json.dump(config_dict, f, indent=4)
    logger.info(f"Configuration saved to {config_path}")

    # Export deployment formats: ONNX (server / ONNX Runtime) and TFLite (Android
    # via ai-edge-torch). These are independent; a failure in one is logged and
    # does not abort the other.
    onnx_path = os.path.join(output_dir, 'nsfw_model.onnx')
    export_to_onnx(output_dir, onnx_path, config)
    export_to_tflite_ai_edge(output_dir, output_dir, config)

In [13]:
def export_to_onnx(model_dir, onnx_path, config):
    logger.info("\nExporting model to ONNX format...")
    try:
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)

        # Create dummy input padded to the configured max length
        dummy_input = tokenizer(
            "sample text",
            return_tensors="pt",
            max_length=config.max_length,
            padding="max_length",
            truncation=True
        )

        os.makedirs(os.path.dirname(onnx_path), exist_ok=True)
        # Pass inputs explicitly (not dict.values()) so the positional order always
        # matches input_names, regardless of what keys the tokenizer emits.
        torch.onnx.export(
            model,
            (dummy_input["input_ids"], dummy_input["attention_mask"]),
            onnx_path,
            export_params=True,
            opset_version=18,  # matches the exporter's implemented opset (avoids failed down-convert)
            do_constant_folding=True,
            input_names=['input_ids', 'attention_mask'],
            output_names=['logits'],
            dynamic_axes={
                'input_ids': {0: 'batch_size'},
                'attention_mask': {0: 'batch_size'},
                'logits': {0: 'batch_size'}
            },
            dynamo=False  # use the stable TorchScript exporter (honors opset + dynamic_axes)
        )
        logger.info(f"ONNX model exported to: {onnx_path}")
        return True
    except Exception as e:
        logger.error(f"ONNX export failed: {e}")
        return False

In [14]:
def export_to_tflite_ai_edge(model_dir, output_dir, config):
    """Convert the fine-tuned PyTorch model to TFLite via litert-torch (LiteRT).

    transformers v5 dropped native TF, and the ONNX -> onnx2tf route produced
    models the LiteRT runtime could not run (LayerNorm MEAN op) or could not
    shrink. litert-torch (formerly 'ai-edge-torch', now renamed) converts
    PyTorch -> LiteRT directly and supports a data-free dynamic-range INT8 recipe.

    Emits:
      - nsfw_model_float32.tflite : reference build (largest)
      - nsfw_model_int8.tflite    : dynamic-range INT8 weights (~1/4 size)
    Both import via Android Studio's ML Model Binding.
    """
    import importlib

    # 'ai-edge-torch' was renamed to 'litert-torch'; the old name is now a stub
    # without .convert. Use whichever module actually provides the API.
    aet = None
    for modname in ('litert_torch', 'ai_edge_torch'):
        try:
            m = importlib.import_module(modname)
        except Exception:
            continue
        if hasattr(m, 'convert'):
            aet = m
            break
    if aet is None:
        logger.error("No working litert_torch/ai_edge_torch (.convert) found; skipping TFLite export.")
        return False
    pkg = aet.__name__
    logger.info(f"Using '{pkg}' for TFLite conversion.")

    try:
        # Reload a clean CPU eval model. eager attention avoids SDPA's
        # data-dependent branches, which torch.export cannot trace.
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir, attn_implementation="eager"
        ).eval().to("cpu")
        tokenizer = AutoTokenizer.from_pretrained(model_dir)

        class _LogitsWrapper(torch.nn.Module):
            """HF returns a dataclass; expose just the logits tensor for tracing."""
            def __init__(self, m):
                super().__init__()
                self.m = m

            def forward(self, input_ids, attention_mask):
                return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

        wrapper = _LogitsWrapper(model).eval()

        # Fixed [1, max_length] sample -> the .tflite takes a fixed-length input.
        enc = tokenizer(
            "sample text", return_tensors="pt",
            max_length=config.max_length, padding="max_length", truncation=True
        )
        sample = (enc["input_ids"], enc["attention_mask"])

        ok = False

        # --- Float32 LiteRT model (reliable baseline) ---
        try:
            logger.info(f"Converting PyTorch -> TFLite (float32) via {pkg}...")
            edge = aet.convert(wrapper, sample)
            f32 = os.path.join(output_dir, "nsfw_model_float32.tflite")
            edge.export(f32)
            logger.info(f"TFLite saved to {f32} ({os.path.getsize(f32) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            logger.error(f"{pkg} float32 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        # --- Dynamic-range INT8 (data-free; ~1/4 size) ---
        try:
            pt2e = importlib.import_module(f"{pkg}.quantize.pt2e_quantizer")
            qcfg = importlib.import_module(f"{pkg}.quantize.quant_config")
            PT2EQuantizer = pt2e.PT2EQuantizer
            get_symmetric_quantization_config = pt2e.get_symmetric_quantization_config
            QuantConfig = qcfg.QuantConfig

            # prepare_pt2e/convert_pt2e moved out of torch.ao into torchao in newer torch.
            prepare_pt2e = convert_pt2e = None
            for modpath in (
                'torchao.quantization.pt2e.quantize_pt2e',
                'torch.ao.quantization.quantize_pt2e',
                'torch.ao.quantization.pt2e.quantize_pt2e',
            ):
                try:
                    qmod = importlib.import_module(modpath)
                    prepare_pt2e, convert_pt2e = qmod.prepare_pt2e, qmod.convert_pt2e
                    break
                except Exception:
                    continue
            if prepare_pt2e is None:
                raise ImportError("prepare_pt2e/convert_pt2e not found (torch.ao or torchao)")

            logger.info(f"Converting PyTorch -> TFLite (dynamic-range INT8) via {pkg} PT2E...")
            quantizer = PT2EQuantizer().set_global(
                get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True)
            )

            # Capture a pre-autograd graph (entry point name varies across torch versions).
            captured = None
            try:
                captured = torch.export.export_for_training(wrapper, sample).module()
            except (AttributeError, TypeError):
                try:
                    from torch._export import capture_pre_autograd_graph
                    captured = capture_pre_autograd_graph(wrapper, sample)
                except Exception:
                    captured = torch.export.export(wrapper, sample).module()

            prepared = prepare_pt2e(captured, quantizer)
            converted = convert_pt2e(prepared)
            edge_q = aet.convert(
                converted, sample, quant_config=QuantConfig(pt2e_quantizer=quantizer)
            )
            i8 = os.path.join(output_dir, "nsfw_model_int8.tflite")
            edge_q.export(i8)
            logger.info(f"TFLite saved to {i8} ({os.path.getsize(i8) / (1024 * 1024):.2f} MB)")
            ok = True
        except Exception as e:
            logger.error(f"{pkg} INT8 conversion failed: {e}")
            import traceback
            traceback.print_exc()

        if ok:
            logger.info("Import in Android Studio via: File > New > Other > TensorFlow Lite Model.")
        else:
            logger.error(f"{pkg} produced no TFLite files.")
        return ok

    except Exception as e:
        logger.error(f"TFLite export failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# INFERENCE

In [15]:
def test_inference(model_dir, config_path=None, test_texts=None, thresholds=None):
    """Test model inference with sample texts using multiple thresholds."""
    if test_texts is None:
        test_texts = [
            "Magandang umaga sa lahat",
            "Kumusta ka",
            "Salamat po",
            "Good morning everyone",
            "inappropriate example",
            "puta",
            "gago",
            "ang ganda mo",
            "bitch ass",
            "who u"
        ]

    logger.info("\nTesting model inference...")

    try:
        # Load model and tokenizer
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model.eval()

        # Load deployment config for the calibrated threshold + max_length
        if config_path is None:
            config_path = os.path.join(model_dir, 'deployment_config.json')
        if os.path.exists(config_path):
            with open(config_path, 'r') as f:
                dcfg = json.load(f)
        else:
            dcfg = {'decision_threshold': 0.5, 'max_length': 128}

        max_length = dcfg.get('max_length', 128)

        # Setup thresholds
        if thresholds is None:
            calibrated = dcfg.get('decision_threshold', 0.5)
            thresholds = {
                'Calibrated': calibrated,
                'Lenient': 0.2,
                'Standard': 0.5,
                'Strict': 0.8
            }

        logger.info("Inference Configuration:")
        for name, val in thresholds.items():
            logger.info(f"  {name} Threshold: {val:.4f}")

        results_data = []

        for text in test_texts:
            inputs = tokenizer(
                text, return_tensors="pt", truncation=True,
                max_length=max_length, padding=True
            )
            with torch.no_grad():
                logits = model(**inputs).logits
                probs = torch.nn.functional.softmax(logits, dim=-1)
                nsfw_prob = probs[0][1].item()

            logger.info("-" * 40)
            logger.info(f"Text: '{text}'")
            logger.info(f"NSFW Probability: {nsfw_prob:.4f}")

            # Store result data
            row = {'text': text, 'nsfw_probability': float(nsfw_prob)}

            # Check against all thresholds
            for name, threshold in thresholds.items():
                label = "NSFW" if nsfw_prob > threshold else "Safe"
                logger.info(f"  [{name} @ {threshold:.2f}] -> {label}")
                row[f'{name}_label'] = label
                row[f'{name}_threshold'] = threshold

            results_data.append(row)

        # Save results to CSV
        if results_data:
            results_df = pd.DataFrame(results_data)
            results_csv_path = os.path.join(model_dir, 'inference_results.csv')
            results_df.to_csv(results_csv_path, index=False)
            logger.info(f"\nInference results saved to {results_csv_path}")

    except Exception as e:
        logger.error(f"Inference test failed: {e}")
        import traceback
        traceback.print_exc()

In [16]:
# ============================================
# Model Trainer Class (Fixed Hyperparameters, No Tuner)
# ============================================

class ModelTrainer:
    def __init__(self, output_dir="models/roberta_tl_nsfw"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        logger.set_log_file(os.path.join(output_dir, 'training.log'))

        self.config = Config()
        self.tokenizer = None
        self.model = None
        self.trainer = None
        self.history = None
        self.train_dataset, self.val_dataset, self.test_dataset = None, None, None
        self.y_train, self.y_val, self.y_test = None, None, None
        self.X_test_raw = None
        self.best_threshold = 0.5

    def prepare_data(self):
        """Step 1: Load and Preprocess Data"""
        logger.section("STEP 1: DATA PREPARATION")
        df = load_dataset()
        (self.train_dataset, self.val_dataset, self.test_dataset,
         self.y_train, self.y_val, self.y_test,
         self.tokenizer, self.X_test_raw) = preprocess_data(df, self.config)
        logger.info("Data preparation complete.")

    def build_model(self):
        """Step 2: Build model with fixed hyperparameters (no tuning)"""
        logger.section("STEP 2: MODEL BUILDING")
        if self.train_dataset is None:
            raise ValueError("Data not loaded. Run prepare_data() first.")
        self.model = build_roberta_model(self.config)

    def train(self):
        """Step 3: Train Model"""
        logger.section("STEP 3: TRAINING")
        if self.model is None:
            raise ValueError("Model not built. Run build_model() first.")

        # Training uses Train/Val
        self.trainer, self.history = train_model(
            self.model, self.train_dataset, self.val_dataset,
            self.tokenizer, self.config, self.output_dir
        )

    def evaluate(self):
        """Step 4: Evaluate Model on Validation Set"""
        logger.section("STEP 4: EVALUATION (VALIDATION SET)")
        if self.trainer is None:
            raise ValueError("Model not available for evaluation.")

        self.accuracy, self.report, self.best_threshold = evaluate_model(
            self.trainer, self.val_dataset, self.y_val, self.output_dir
        )
        logger.info(f"Final validation accuracy: {self.accuracy:.4f}")

    def test(self):
        """Step 5: Evaluate on Test Set (Unseen Data)"""
        logger.section("STEP 5: TEST SET EVALUATION")
        if self.test_dataset is None:
            raise ValueError("Test data not loaded.")
        if self.trainer is None:
            raise ValueError("Model not available.")

        test_acc, test_report, _ = evaluate_model(
            self.trainer, self.val_dataset, self.y_val, self.output_dir,
            test_dataset=self.test_dataset, y_test=self.y_test,
            threshold=self.best_threshold  # reuse threshold calibrated in evaluate()
        )
        logger.info(f"Final Test Set Accuracy: {test_acc:.4f}")

    def export(self):
        """Step 6: Export Artifacts"""
        logger.section("STEP 6: EXPORT")
        if self.history:
            plot_training_history(self.history, self.output_dir)

        save_artifacts(self.trainer, self.tokenizer, self.config, self.output_dir, self.best_threshold)
        logger.info(f"Pipeline complete. Artifacts in {self.output_dir}")
        return self.output_dir

    def run_all(self):
        """Run the full pipeline sequentially"""
        self.prepare_data()
        self.build_model()
        self.train()
        self.evaluate()
        self.test()  # Explicit Test Step
        return self.export()

In [17]:
if __name__ == "__main__":
    try:
        # Initialize Trainer
        trainer_pipeline = ModelTrainer()

        # Execute Pipeline Steps
        trainer_pipeline.prepare_data()
        trainer_pipeline.build_model()
        trainer_pipeline.train()
        trainer_pipeline.evaluate()
        trainer_pipeline.test()  # Evaluate on held-out test set
        output_dir = trainer_pipeline.export()

        # Test inference with actual test data samples
        print("\nRunning inference test on Test Set samples...")
        if trainer_pipeline.X_test_raw is not None and len(trainer_pipeline.X_test_raw) > 0:
            sample_texts = list(trainer_pipeline.X_test_raw[:20])
        else:
            print("Warning: No test data found. Using default samples.")
            sample_texts = None

        test_inference(model_dir=output_dir, test_texts=sample_texts)

    except KeyboardInterrupt:
        print("\nTraining interrupted by user.")
    except Exception as e:
        print(f"Error during execution: {e}")
        import traceback
        traceback.print_exc()

    print("\nProgram execution completed.")

04:08:58 - Logging to file: models/roberta_tl_nsfw/training.log
04:08:58 - 
04:08:58 - MODEL CONFIGURATION
04:08:58 - ======================================================================
04:08:58 - Model Version: 1.0.0
04:08:58 - Base Model: dost-asti/RoBERTa-tl-sentiment-analysis
04:08:58 - Max Sequence Length: 128
04:08:58 - Batch Size: 16
04:08:58 - Epochs: 5
04:08:58 - Learning Rate: 2e-05
04:08:58 - Weight Decay: 0.01
04:08:58 - Validation Split: 0.15
04:08:58 - Test Split: 0.15
04:08:58 - 
04:08:58 - STEP 1: DATA PREPARATION
04:08:58 - ======================================================================
04:08:58 - Loading dataset from cleaned_data.csv...
04:08:58 - Dataset loaded successfully. Shape: (5579, 2)
04:08:58 - Mapping text labels (safe/nsfw) to binary integers...
04:08:59 - Dataset validation complete. Clean shape: (5579, 2)
04:08:59 - Label distribution:
label
0    3079
1    2500
Name: count, dtype: int64
04:08:59 - Preprocessing data...
04:08:59 - Training sample

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/469k [00:00<?, ?B/s]

...

(00:20) [ FAIL] LiteRT-Torch Convert

04:12:32 - litert_torch INT8 conversion failed: Failed to run converter passes: /usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134:0: error: 'stablehlo.uniform_dequantize' op operand #0 must be ranked tensor of per-tensor integer quantized or per-axis integer quantized values, but got 'tensor<768x768xi8>'

04:12:32 - Import in Android Studio via: File > New > Other > TensorFlow Lite Model.
04:12:32 - Pipeline complete. Artifacts in models/roberta_tl_nsfw

Running inference test on Test Set samples...
04:12:32 - 
Testing model inference...


Traceback (most recent call last):
  File "/tmp/ipykernel_569/2714650129.py", line 116, in export_to_tflite_ai_edge
    edge_q = aet.convert(
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/interface.py", line 323, in convert
    return Converter().convert(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/interface.py", line 208, in convert
    converted_model = core.convert_signatures(
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/contextlib.py", line 81, in inner
    return func(*args, **kwds)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/core.py", line 158, in convert_signatures
    exporter = litert_converter.exported_programs_to_flatbuffer(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/litert_torch/_convert/litert_converter.py", line

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

04:12:33 - Inference Configuration:
04:12:33 -   Calibrated Threshold: 0.5000
04:12:33 -   Lenient Threshold: 0.2000
04:12:33 -   Standard Threshold: 0.5000
04:12:33 -   Strict Threshold: 0.8000
04:12:33 - ----------------------------------------
04:12:33 - Text: 'hanapin ang langgam excuse my hairy pit'
04:12:33 - NSFW Probability: 0.5139
04:12:33 -   [Calibrated @ 0.50] -> NSFW
04:12:33 -   [Lenient @ 0.20] -> NSFW
04:12:33 -   [Standard @ 0.50] -> NSFW
04:12:33 -   [Strict @ 0.80] -> Safe
04:12:33 - ----------------------------------------
04:12:33 - Text: 'she got fire ass'
04:12:33 - NSFW Probability: 0.9947
04:12:33 -   [Calibrated @ 0.50] -> NSFW
04:12:33 -   [Lenient @ 0.20] -> NSFW
04:12:33 -   [Standard @ 0.50] -> NSFW
04:12:33 -   [Strict @ 0.80] -> NSFW
04:12:33 - ----------------------------------------
04:12:33 - Text: 'stress posting ulit kasi wym di pa rin ma balance na adjust ko naman na lahat'
04:12:33 - NSFW Probability: 0.0138
04:12:33 -   [Calibrated @ 0.50] -> Saf

# TFLITE PARITY CHECK (INT8 / float32 vs PyTorch)

In [18]:
def tflite_parity_check(output_dir=None, max_samples=None):
    """Run the held-out TEST set through each exported .tflite and compare against
    the PyTorch reference (test_predictions.csv) at the calibrated threshold.

    Reports, per model: TFLite accuracy vs the PyTorch model, agreement rate,
    label flips, and probability drift -- so INT8/float32 deployment can be trusted.
    """
    import importlib

    # --- Resolve output dir + the raw test texts (same order as test_predictions.csv) ---
    try:
        texts = list(trainer_pipeline.X_test_raw)
        if output_dir is None:
            output_dir = trainer_pipeline.output_dir
    except NameError:
        # Fallback: reconstruct deterministically (same seed/split) if run standalone
        if output_dir is None:
            output_dir = "models/roberta_tl_nsfw"
        df = load_dataset()
        *_, texts = preprocess_data(df, Config())
        texts = list(texts)

    if max_samples is not None:
        texts = texts[:max_samples]

    # --- PyTorch reference predictions ---
    ref_path = os.path.join(output_dir, 'test_predictions.csv')
    if not os.path.exists(ref_path):
        logger.error(f"Reference {ref_path} not found; run the test step first.")
        return
    ref = pd.read_csv(ref_path)
    if max_samples is not None:
        ref = ref.iloc[:max_samples]
    y_true = ref['y_true'].to_numpy()
    pt_prob = ref['y_prob'].to_numpy()
    pt_pred = ref['y_pred'].to_numpy()

    # --- Threshold + tokenizer ---
    cfg_path = os.path.join(output_dir, 'deployment_config.json')
    threshold = 0.5
    max_length = 128
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            dcfg = json.load(f)
        threshold = dcfg.get('decision_threshold', 0.5)
        max_length = dcfg.get('max_length', 128)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)

    # litert-torch targets the LiteRT runtime; prefer its interpreter, fall back to tf.lite.
    Interpreter = None
    for mod in ('ai_edge_litert.interpreter', 'litert.interpreter'):
        try:
            Interpreter = importlib.import_module(mod).Interpreter
            break
        except Exception:
            continue
    if Interpreter is None:
        from tensorflow.lite import Interpreter

    def _input_detail(in_details, key, fallback_idx):
        for d in in_details:
            if key in d['name']:
                return d
        return in_details[fallback_idx]

    def _run_tflite(model_path):
        interp = Interpreter(model_path=model_path)
        in_details = interp.get_input_details()
        id_d = _input_detail(in_details, 'input_ids', 0)
        mask_d = _input_detail(in_details, 'attention_mask', 1)

        # Only resize when the model's declared shape differs from [1, max_length].
        want = [1, max_length]
        if list(id_d['shape']) != want:
            interp.resize_tensor_input(id_d['index'], want)
        if list(mask_d['shape']) != want:
            interp.resize_tensor_input(mask_d['index'], want)
        interp.allocate_tensors()

        # Re-fetch details: indices/dtypes/shapes can change after allocate_tensors().
        in_details = interp.get_input_details()
        out_details = interp.get_output_details()
        id_d = _input_detail(in_details, 'input_ids', 0)
        mask_d = _input_detail(in_details, 'attention_mask', 1)

        probs = []
        for text in texts:
            enc = tokenizer(text, truncation=True, max_length=max_length,
                            padding='max_length', return_tensors='np')
            interp.set_tensor(id_d['index'], enc['input_ids'].astype(id_d['dtype']))
            interp.set_tensor(mask_d['index'], enc['attention_mask'].astype(mask_d['dtype']))
            interp.invoke()
            logits = interp.get_tensor(out_details[0]['index'])[0]
            e = np.exp(logits - np.max(logits))
            probs.append(float((e / e.sum())[1]))   # P(NSFW)
        return np.array(probs)

    logger.section("STEP 7: TFLITE PARITY CHECK")
    logger.info(f"PyTorch reference: {len(texts)} test samples, threshold {threshold:.4f}")
    pt_acc = float(np.mean(pt_pred == y_true))
    logger.info(f"PyTorch test accuracy (reference): {pt_acc:.4f}")

    rows = []
    for name in ('int8', 'float32'):
        path = os.path.join(output_dir, f'nsfw_model_{name}.tflite')
        if not os.path.exists(path):
            continue
        logger.info(f"\nEvaluating {os.path.basename(path)} ...")
        try:
            tfl_prob = _run_tflite(path)
        except Exception as e:
            logger.error(f"  Interpreter run failed for {name}: {e}")
            continue
        tfl_pred = (tfl_prob > threshold).astype(int)

        tfl_acc = float(np.mean(tfl_pred == y_true))
        agreement = float(np.mean(tfl_pred == pt_pred))
        flips = int(np.sum(tfl_pred != pt_pred))
        mean_abs = float(np.mean(np.abs(tfl_prob - pt_prob)))
        max_abs = float(np.max(np.abs(tfl_prob - pt_prob)))

        logger.info(f"  TFLite accuracy:        {tfl_acc:.4f}  (PyTorch {pt_acc:.4f}, delta {tfl_acc - pt_acc:+.4f})")
        logger.info(f"  Agreement w/ PyTorch:   {agreement:.4f}  ({flips} label flips / {len(texts)})")
        logger.info(f"  Prob drift |tfl-pt|:    mean {mean_abs:.4f}, max {max_abs:.4f}")
        rows.append({
            'model': name, 'tflite_accuracy': tfl_acc, 'pytorch_accuracy': pt_acc,
            'accuracy_delta': tfl_acc - pt_acc, 'agreement': agreement,
            'label_flips': flips, 'n_samples': len(texts),
            'mean_abs_prob_diff': mean_abs, 'max_abs_prob_diff': max_abs,
            'threshold': threshold,
        })

    if rows:
        out_csv = os.path.join(output_dir, 'tflite_parity.csv')
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        logger.info(f"\nParity report saved to {out_csv}")
    else:
        logger.warning("No .tflite models ran successfully. Check the export step / interpreter errors above.")


# Run the parity check on the exported TFLite models before zipping/downloading.
try:
    tflite_parity_check()
except Exception as e:
    print(f"TFLite parity check skipped: {e}")
    import traceback
    traceback.print_exc()

04:12:36 - 
04:12:36 - STEP 7: TFLITE PARITY CHECK
04:12:36 - ======================================================================
04:12:36 - PyTorch reference: 837 test samples, threshold 0.5000
04:12:36 - PyTorch test accuracy (reference): 0.8973
04:12:36 - 
Evaluating nsfw_model_float32.tflite ...
04:16:48 -   TFLite accuracy:        0.8973  (PyTorch 0.8973, delta +0.0000)
04:16:48 -   Agreement w/ PyTorch:   1.0000  (0 label flips / 837)
04:16:48 -   Prob drift |tfl-pt|:    mean 0.0000, max 0.0007
04:16:48 - 
Parity report saved to models/roberta_tl_nsfw/tflite_parity.csv


# DOWNLOAD ARTIFACTS

In [19]:
import os
from google.colab import files

output_zip = 'Dost_roberta.zip'
folders = ['models']

# Filter only existing folders
folders_to_zip = [f for f in folders if os.path.exists(f)]

if folders_to_zip:
    print(f"Zipping: {', '.join(folders_to_zip)}")
    cmd = f"zip -r {output_zip} {' '.join(folders_to_zip)}"
    os.system(cmd)

    if os.path.exists(output_zip):
        print(f"Downloading {output_zip} ({os.path.getsize(output_zip)/1e6:.2f} MB)...")
        files.download(output_zip)
    else:
        print("Error creating zip file.")
else:
    print("No folders found to zip.")

Zipping: models


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>